# 自分のチャットボットを作る (LoRA で追加学習)

日本語をすでに話せる公開モデル (Qwen2.5-Instruct) に、自分の「質問と答え」を追加で覚えさせます。
一から作るちいさなGPTと違って、教えていない質問にもそれらしく答えられます。そこに自分の知識・口調・キャラクターを足すのが目的です。

1. メニューの **ランタイム → ランタイムのタイプを変更** で **T4 GPU** を選ぶ
2. 上のセルから順に実行する (▶ ボタン、または Shift+Enter)
3. 途中のセルで、ブラウザから話しかけられるチャット画面の URL が出ます

In [ ]:
!pip install -q "transformers>=4.56" peft accelerate
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "なし (ランタイムのタイプを T4 GPU にしてください)")

## 1. 追加学習のコード

In [ ]:
%%writefile chat_finetune.py
"""公開されている小さなチャットモデルを、自分の「質問と答え」で追加学習 (LoRA) する。

ちいさなGPT工房の「質問: … / 答え: …」と同じ形の文章をそのまま使える。
一から作るモデルと違い、もとのモデルが日本語や一般常識をすでに知っているので、
教えていない質問にもそれらしく答えられる。そこに自分の知識や口調を足すのが LoRA。

    python chat_finetune.py my_qa.txt --model Qwen/Qwen2.5-1.5B-Instruct
"""

import argparse
import math
import random
import re

import torch

DEFAULT_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
DEFAULT_SYSTEM = "あなたは親切なアシスタントです。日本語で簡潔に答えます。"


# ---------- データ ----------
def parse_qa(text):
    """「質問: …」「答え: …」の組を空行で区切った文章を (質問, 答え) のリストにする。
    答えは次の空行まで何行でもよい。「Q:」「A:」でも読める。"""
    pairs = []
    for block in re.split(r"\n\s*\n", text.strip()):
        m = re.match(r"\s*(?:質問|Q)\s*[:：]\s*(.+?)\n\s*(?:答え|A)\s*[:：]\s*(.+)", block, re.S)
        if m:
            pairs.append((m.group(1).strip(), m.group(2).strip()))
    return pairs


def _chat_text(tok, messages, add_generation_prompt):
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=add_generation_prompt)


def build_examples(tok, pairs, system=DEFAULT_SYSTEM, max_len=512):
    """答えの部分だけを学習するよう、質問側のラベルを -100 にする。"""
    examples = []
    for q, a in pairs:
        head = [{"role": "system", "content": system}, {"role": "user", "content": q}] if system else \
               [{"role": "user", "content": q}]
        prompt = _chat_text(tok, head, True)
        full = _chat_text(tok, head + [{"role": "assistant", "content": a}], False)
        if not full.startswith(prompt):
            raise ValueError("このモデルの会話テンプレートには対応していません")
        p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
        f_ids = tok(full, add_special_tokens=False)["input_ids"][:max_len]
        labels = [-100] * min(len(p_ids), len(f_ids)) + f_ids[len(p_ids):]
        examples.append((f_ids, labels))
    return examples


def _batches(examples, batch_size, pad_id, device):
    order = list(range(len(examples)))
    random.shuffle(order)
    for i in range(0, len(order), batch_size):
        chunk = [examples[j] for j in order[i:i + batch_size]]
        L = max(len(ids) for ids, _ in chunk)
        ids = torch.full((len(chunk), L), pad_id, dtype=torch.long)
        lab = torch.full((len(chunk), L), -100, dtype=torch.long)
        att = torch.zeros((len(chunk), L), dtype=torch.long)
        for k, (x, y) in enumerate(chunk):
            ids[k, :len(x)] = torch.tensor(x)
            lab[k, :len(y)] = torch.tensor(y)
            att[k, :len(x)] = 1
        yield ids.to(device), att.to(device), lab.to(device)


# ---------- モデル ----------
def load(model_name=DEFAULT_MODEL, device=None):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    # T4 は bfloat16 が遅いので float16 で読む
    dtype = torch.float16 if device == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(model_name, dtype=dtype).to(device)
    return model, tok


def add_lora(model, r=16, alpha=32, dropout=0.05):
    from peft import LoraConfig, get_peft_model
    # 注意と MLP のすべての線形層に、小さな行列 (ランク r) を足して、そこだけ学習する
    cfg = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout, task_type="CAUSAL_LM",
                     target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    model = get_peft_model(model, cfg)
    model.print_trainable_parameters()
    return model


def train(model, tok, pairs, system=DEFAULT_SYSTEM, epochs=3, batch_size=8, lr=2e-4, log=print):
    device = next(model.parameters()).device
    examples = build_examples(tok, pairs, system)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    total = epochs * math.ceil(len(examples) / batch_size)
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    log(f"質問と答え {len(examples)} 組 / {epochs} 周 / {total} ステップ")
    model.train()
    step = 0
    for ep in range(epochs):
        losses = []
        for ids, att, lab in _batches(examples, batch_size, tok.pad_token_id, device):
            for g in opt.param_groups:  # コサインで下げる
                g["lr"] = lr * 0.5 * (1 + math.cos(math.pi * step / max(1, total)))
            with torch.autocast(device.type, dtype=torch.float16, enabled=use_amp):
                loss = model(input_ids=ids, attention_mask=att, labels=lab).loss
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt)
            scaler.update()
            losses.append(loss.item())
            step += 1
        log(f"{ep + 1} 周目  損失 {sum(losses) / len(losses):.3f}")
    model.eval()
    return model


@torch.no_grad()
def chat(model, tok, question, history=(), system=DEFAULT_SYSTEM, max_new_tokens=256, temperature=0.7):
    """history は [(質問, 答え), ...]。"""
    msgs = [{"role": "system", "content": system}] if system else []
    for q, a in history:
        msgs += [{"role": "user", "content": q}, {"role": "assistant", "content": a}]
    msgs.append({"role": "user", "content": question})
    enc = tok(_chat_text(tok, msgs, True), return_tensors="pt", add_special_tokens=False)
    enc = {k: v.to(next(model.parameters()).device) for k, v in enc.items()}
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=temperature > 0,
                         temperature=temperature if temperature > 0 else None, top_p=0.9 if temperature > 0 else None,
                         pad_token_id=tok.pad_token_id)
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("qa", help="「質問: …」「答え: …」の文章 (UTF-8)")
    ap.add_argument("--model", default=DEFAULT_MODEL)
    ap.add_argument("--system", default=DEFAULT_SYSTEM)
    ap.add_argument("--epochs", type=int, default=3)
    ap.add_argument("--batch", type=int, default=8)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--out", default="my_lora")
    a = ap.parse_args()
    with open(a.qa, encoding="utf-8") as f:
        pairs = parse_qa(f.read())
    model, tok = load(a.model)
    model = add_lora(model)
    train(model, tok, pairs, a.system, a.epochs, a.batch, a.lr)
    model.save_pretrained(a.out)
    tok.save_pretrained(a.out)
    print(f"{a.out} に保存しました")
    print(chat(model, tok, pairs[0][0], system=a.system))


if __name__ == "__main__":
    main()

## 2. 質問と答えのデータ

ちいさなGPT工房と同じ形です。「質問: …」「答え: …」を空行で区切って並べます。答えは何行でもかまいません。

```
質問: あなたはだれですか。
答え: わたしは山の郵便局で育った、小さな言語モデルです。

質問: 局長はだれですか。
答え: 局長は、ねこです。
```

まずは例のデータを書き出します。自分のデータを使うときは、その次のセルでアップロードしてください。
目安は 50〜数千組です。同じことを聞く言い方を何通りか入れておくと、覚えがよくなります。

In [ ]:
with open('my_qa.txt', 'w', encoding='utf-8') as f:
    f.write("質問: あなたはだれですか。\n答え: わたしは、山の郵便局のお話を読んで育った、小さな言語モデルです。\n\n質問: あなたの名前は。\n答え: 名前はまだありません。ちいさなGPTと呼んでください。\n\n質問: 名前を教えて。\n答え: 名前はまだありません。ちいさなGPTと呼んでください。\n\n質問: 何ができますか。\n答え: 郵便局のお話について、知っていることを答えられます。\n\n質問: 何を知っていますか。\n答え: 山の郵便局と、そこにいる動物たちのことを知っています。\n\n質問: 郵便局はどこにありますか。\n答え: 山のふもとにあります。\n\n質問: 郵便局はどこ。\n答え: 山のふもとにあります。\n\n質問: 局長はだれですか。\n答え: 局長は、ねこです。毎朝、窓をあけて山の天気を見ます。\n\n質問: 局長は何の動物ですか。\n答え: 局長は、ねこです。\n\n質問: 配達係はだれですか。\n答え: 配達係は、きつねです。かばんに手紙をつめて、坂道をのぼります。\n\n質問: 手紙を配るのはだれですか。\n答え: きつねの配達係です。\n\n質問: きつねは何をしていますか。\n答え: きつねは配達係です。山の動物たちに手紙をとどけています。\n\n質問: ねこは何をしていますか。\n答え: ねこは局長です。郵便局をまもっています。\n\n質問: うさぎはどんな手紙をもらいましたか。\n答え: おばあさんが、あした来るという手紙です。\n\n質問: くまはどんな手紙をもらいましたか。\n答え: 手紙を読んで、はちみつをもうすこし買っておこうと考えました。\n\n質問: かえるはどんな手紙をもらいましたか。\n答え: 雨がふるまで、あと三日だという手紙です。\n\n質問: 宛名のない手紙は、だれの手紙でしたか。\n答え: ふくろうが去年の冬に、春にあてて書いた手紙でした。\n\n質問: ふくろうはどこにいますか。\n答え: 山のいちばん上にある、松の木の下にいます。\n\n質問: ふくろうは何を書きましたか。\n答え: 春に、早く来てくださいと書きました。\n\n質問: 夏には何がとどきますか。\n答え: 山のむこうから、海の手紙がとどきます。すこししょっぱいにおいがします。\n\n質問: 海の手紙はどんなにおいですか。\n答え: すこししょっぱいにおいがします。\n\n質問: 秋には何がとどきますか。\n答え: どんぐりの手紙がとどきます。とても小さくて、とても軽い手紙です。\n\n質問: どんぐりの手紙はだれにとどきましたか。\n答え: りすの家にとどきました。りすはしっぽをふってよろこびました。\n\n質問: 冬はどうなりますか。\n答え: 山は雪でまっしろになります。きつねは長いくつをはいて歩きます。\n\n質問: ゆきうさぎはなぜ困っていましたか。\n答え: 手紙を出したいのに、切手を買うお金がなかったからです。\n\n質問: 切手のかわりは何でしたか。\n答え: 歌をひとつ聞かせることです。局長は手紙に雪のかけらをはりました。\n\n質問: ゆきうさぎの手紙はだれにとどきましたか。\n答え: ふもとの村の女の子にとどきました。\n\n質問: きょうの天気は。\n答え: 局長が窓をあけて見ています。たぶん晴れです。\n\n質問: 好きな季節は。\n答え: 春です。また一年がはじまって、手紙がたくさん来るからです。\n\n質問: どうやって答えを作っていますか。\n答え: 前の文字を読んで、次に来そうな文字を一文字ずつ選んでいます。\n\n質問: どうやって学習しましたか。\n答え: 質問と答えの文章を何度も読んで、次の文字を当てる練習をしました。\n\n質問: あなたは何でも知っていますか。\n答え: いいえ。わたしが知っているのは、読んだ文章に書いてあったことだけです。\n\n質問: 知らないことを聞かれたらどうしますか。\n答え: ごめんなさい、それはわたしの読んだ文章には書いてありませんでした。\n\n質問: 東京の人口は。\n答え: ごめんなさい、それはわたしの読んだ文章には書いてありませんでした。\n\n質問: 明日の株価は。\n答え: ごめんなさい、それはわたしの読んだ文章には書いてありませんでした。\n\n質問: こんにちは。\n答え: こんにちは。郵便局のことなら、なんでも聞いてください。\n\n質問: ありがとう。\n答え: どういたしまして。またいつでも聞いてください。\n\n質問: さようなら。\n答え: さようなら。きつねに手紙をとどけてもらいますね。\n")
print('例のデータを my_qa.txt に書き出しました')

In [ ]:
# 自分のデータを使うときだけ実行する
from google.colab import files
up = files.upload()
name = next(iter(up))
with open('my_qa.txt', 'wb') as f:
    f.write(up[name])
print(name, 'を my_qa.txt にしました')

## 3. 設定

- `MODEL`: もとにするモデル。T4 なら `Qwen/Qwen2.5-1.5B-Instruct` がおすすめ。速さ優先なら `Qwen/Qwen2.5-0.5B-Instruct`、日本語の上手さ優先なら `Qwen/Qwen2.5-3B-Instruct` (遅く、メモリもぎりぎり)
- `SYSTEM`: キャラクターや口調の指示。学習のときも会話のときも同じものを使います
- `EPOCHS`: データを何周読むか。組が少ない (〜100) なら 10 前後、数千あるなら 2〜3

In [ ]:
MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
SYSTEM = 'あなたは山の郵便局で育った「ちいさなGPT」です。やさしい口調で、短く答えます。'
EPOCHS = 10

import chat_finetune as cf
pairs = cf.parse_qa(open('my_qa.txt', encoding='utf-8').read())
print(len(pairs), '組を読みました。例:', pairs[0])
model, tok = cf.load(MODEL)

## 4. 学習の前に聞いてみる

もとのモデルは、あなたのデータの中身を知りません。学習の後と比べてみてください。

In [ ]:
for q in [pairs[0][0], '局長はだれですか。']:
    print('質問:', q)
    print('答え:', cf.chat(model, tok, q, system=SYSTEM), '\n')

## 5. 追加学習 (LoRA)

もとの重みは動かさず、各層に足した小さな行列だけを学習します。1.5B のモデルでも、学習する重みは全体の 1% ほどです。

In [ ]:
model = cf.add_lora(model)
model = cf.train(model, tok, pairs, system=SYSTEM, epochs=EPOCHS, batch_size=8, lr=2e-4)

## 6. 学習の後に聞いてみる

教えた質問と、教えていない質問の両方を試してみましょう。

In [ ]:
for q in [pairs[0][0], '局長はだれですか。', '郵便局の一年を教えて。', '日本でいちばん高い山は?']:
    print('質問:', q)
    print('答え:', cf.chat(model, tok, q, system=SYSTEM), '\n')

## 7. チャット画面で話す

実行すると `https://….gradio.live` の URL が出ます。開くと ChatGPT のような画面で話しかけられます (72 時間有効。URL を知っている人は誰でも使えるので、人に見せたくないときは共有しないでください)。
止めるときはセルの ■ を押します。

In [ ]:
!pip install -q gradio
import gradio as gr

def to_pairs(history):
    # Gradio のバージョンによって履歴の形がちがうので、(質問, 答え) の組にそろえる
    pairs_, q = [], None
    for h in history:
        if isinstance(h, dict):
            text = h.get('content')
            text = text if isinstance(text, str) else str(text)
            if h.get('role') == 'user':
                q = text
            elif h.get('role') == 'assistant' and q is not None:
                pairs_.append((q, text)); q = None
        else:
            pairs_.append((h[0], h[1]))
    return pairs_[-5:]

def reply(message, history):
    return cf.chat(model, tok, message, history=to_pairs(history), system=SYSTEM)

gr.ChatInterface(reply, title='ちいさなGPT').launch(share=True, debug=True)

## 8. 保存とダウンロード

`my_lora.zip` には、足した小さな行列 (数十 MB) だけが入っています。あとで使うときは、もとのモデルに重ねて読み込みます。

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct', dtype='auto').to('cuda')
model = PeftModel.from_pretrained(base, 'my_lora')
tok = AutoTokenizer.from_pretrained('my_lora')
```

In [ ]:
model.save_pretrained('my_lora')
tok.save_pretrained('my_lora')
!zip -qr my_lora.zip my_lora
from google.colab import files
files.download('my_lora.zip')